In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
load_dotenv()
DB_URL = os.getenv("DB_URL")
engine = create_engine(DB_URL)
print("Database engine created successfully.")

Database engine created successfully.


**` Read every table from the database and look at each one on its own`**

In [3]:
customers = pd.read_sql("SELECT * FROM customers", engine)
orders = pd.read_sql("SELECT * FROM orders", engine)
products = pd.read_sql("SELECT * FROM products", engine)
sellers = pd.read_sql("SELECT * FROM sellers", engine)
geolocation = pd.read_sql("SELECT * FROM geolocation", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
order_payments = pd.read_sql("SELECT * FROM order_payments", engine)
order_reviews = pd.read_sql("SELECT * FROM order_reviews", engine)
product_category_translation = pd.read_sql("SELECT * FROM product_category_translation",engine) 
print("All Olist tables loaded successfully.")

All Olist tables loaded successfully.


**`Check row counts, keys, duplicates, and what one row means in each table`**

In [4]:
tables = {
    "orders": orders,
    "customers": customers,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "product_category_translation": product_category_translation}

summary = []
for name, df in tables.items():
    summary.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum()})

table_summary = pd.DataFrame(summary)
table_summary

,table,rows,columns,duplicate_rows
0,orders,99441,8,0
1,customers,99441,5,0
2,order_items,112650,7,0
3,order_payments,103886,5,0
4,order_reviews,99224,7,0
5,products,32951,9,0
6,sellers,3095,4,0
7,geolocation,1000163,5,261831
8,product_category_translation,71,2,0


In [5]:
for name, df in tables.items():
    print(f"{name}")
    print(list(df.columns))

orders
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
customers
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
order_items
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
order_payments
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
order_reviews
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
products
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
sellers
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
geolocation
['geolocatio

In [6]:
key_checks = {
    "orders.order_id": orders["order_id"].nunique() == len(orders),
    "customers.customer_id": customers["customer_id"].nunique() == len(customers),
    "products.product_id": products["product_id"].nunique() == len(products),
    "sellers.seller_id": sellers["seller_id"].nunique() == len(sellers)}
pd.Series(key_checks, name="is_unique")

orders.order_id          True
customers.customer_id    True
products.product_id      True
sellers.seller_id        True
Name: is_unique, dtype: bool

## `Table Grain`  

- orders: one row represents one order.  
- customers: one row represents one customer record.  
- order_items: one row represents one item within an order.  
- order_payments: one row represents one payment record for an order.  
- order_reviews: one row represents one review record.  
- products: one row represents one product.  
- sellers: one row represents one seller.  
- geolocation: one row represents one geolocation record for a ZIP-code prefix.  
- product_category_translation: one row represents one product category translation.  

**` Aggregate before you join — order items and payments have many rows per order`**

In [7]:
# Aggregation - Items
items_agg = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        avg_item_price=("price", "mean")).reset_index())
items_agg.head()

,order_id,item_count,unique_products,unique_sellers,total_price,total_freight,avg_item_price
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93,239.90
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14,199.90


In [8]:
assert items_agg["order_id"].is_unique
print("Unique orders in aggregated items:", items_agg["order_id"].nunique())
print("Rows in aggregated items:", len(items_agg))

Unique orders in aggregated items: 98666
Rows in aggregated items: 98666


In [9]:
# Aggregation - payments
payments_agg = (
    order_payments
    .groupby("order_id")
    .agg(
        payment_count=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max")).reset_index())
payments_agg.head()

,order_id,payment_count,total_payment_value,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3


In [16]:
assert payments_agg["order_id"].is_unique
print("Unique orders in aggregated payments:", payments_agg["order_id"].nunique())
print("Rows in aggregated payments:", len(payments_agg))

Unique orders in aggregated payments: 99440
Rows in aggregated payments: 99440


In [17]:
orders_ml = orders.merge(
    items_agg,
    on="order_id",
    how="left")

orders_ml = orders_ml.merge(
    payments_agg,
    on="order_id",
    how="left")

print(orders_ml.shape)
orders_ml.head()

(99441, 17)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,unique_products,unique_sellers,total_price,total_freight,avg_item_price,payment_count,total_payment_value,max_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,1.0,1.0,29.99,8.72,29.99,3.0,38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,1.0,1.0,118.70,22.76,118.70,1.0,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,1.0,1.0,159.90,19.22,159.90,1.0,179.12,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,1.0,1.0,45.00,27.20,45.00,1.0,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,1.0,1.0,19.90,8.72,19.90,1.0,28.62,1.0


In [18]:
# Join - Customers
print("Orders:", orders_ml["order_id"].nunique())
print("Customers:", customers["customer_id"].nunique())
print("Customer rows:", len(customers))

orders_ml = orders_ml.merge(
    customers,
    on="customer_id",
    how="left")
print(orders_ml.shape)

Orders: 99441
Customers: 99441
Customer rows: 99441
(99441, 21)


In [19]:
# Verify ONE ROW PER ORDER
print("Rows:", len(orders_ml))
print("Unique orders:", orders_ml["order_id"].nunique())

duplicates = orders_ml["order_id"].duplicated().sum()
print("Duplicate order_id:", duplicates)

Rows: 99441
Unique orders: 99441
Duplicate order_id: 0


In [20]:
# Save ML Table as Artifact
from pathlib import Path
Path("artifacts").mkdir(exist_ok=True)
orders_ml.to_csv(
    "artifacts/orders_ml.csv",
    index=False)
print("orders_ml artifact saved successfully.")

orders_ml artifact saved successfully.
